In [25]:
from langchain_core.documents import  Document

documents = [
    Document(
        page_content = "Dogs are great companion, known for their loyalty and friendliness",
        metadata ={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content = "Cats are independent pets that often enjoy their own space. ",
        metadata={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content = "GoldFish are popular pets for beginners, requiring relatively simple care.",
        metadata = {"source":"fish-pets-doc"}
    ),
    Document(
        page_content = "Parrots are intelligent birds capable of mimicking human speech.",
        metadata = {"source":"bird-pets-doc"}
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around",
        metadata = {"source":"mammal-pets-doc"}
    )
]

In [26]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companion, known for their loyalty and friendliness'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space. '),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='GoldFish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around')]

In [27]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key = groq_api_key,model="openai/gpt-oss-120b" ) 
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7748624db700>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x774867b90b20>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [28]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 311.92it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
## VectorStores
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents,embedding=embeddings)


In [30]:
vectorstore.similarity_search("cat")

[Document(id='92bc8a41-b48c-40b8-93be-6289cacd4a77', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space. '),
 Document(id='0ab590c5-c554-4c95-ac8f-fad8cc54345a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space. '),
 Document(id='a0ee7c84-10f2-47ae-807f-f6772f8d7b13', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companion, known for their loyalty and friendliness'),
 Document(id='af36ebce-41cc-4e8d-a492-c2e18b558e33', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companion, known for their loyalty and friendliness')]

In [31]:
## Async query
await vectorstore.asimilarity_search("cat")

[Document(id='92bc8a41-b48c-40b8-93be-6289cacd4a77', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space. '),
 Document(id='0ab590c5-c554-4c95-ac8f-fad8cc54345a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space. '),
 Document(id='a0ee7c84-10f2-47ae-807f-f6772f8d7b13', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companion, known for their loyalty and friendliness'),
 Document(id='af36ebce-41cc-4e8d-a492-c2e18b558e33', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companion, known for their loyalty and friendliness')]

In [32]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='92bc8a41-b48c-40b8-93be-6289cacd4a77', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space. '),
  0.935105562210083),
 (Document(id='0ab590c5-c554-4c95-ac8f-fad8cc54345a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space. '),
  0.935105562210083),
 (Document(id='a0ee7c84-10f2-47ae-807f-f6772f8d7b13', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companion, known for their loyalty and friendliness'),
  1.5351176261901855),
 (Document(id='af36ebce-41cc-4e8d-a492-c2e18b558e33', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companion, known for their loyalty and friendliness'),
  1.5351176261901855)]

In [33]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='92bc8a41-b48c-40b8-93be-6289cacd4a77', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space. ')],
 [Document(id='af36ebce-41cc-4e8d-a492-c2e18b558e33', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companion, known for their loyalty and friendliness')]]

In [34]:
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='92bc8a41-b48c-40b8-93be-6289cacd4a77', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space. ')],
 [Document(id='af36ebce-41cc-4e8d-a492-c2e18b558e33', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companion, known for their loyalty and friendliness')]]

In [35]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only


{question}

Context:
{context}


"""

prompt = ChatPromptTemplate.from_messages([("human",message)])

rag_chain = {"context":retriever,"question":RunnablePassthrough()} | prompt | llm


response = rag_chain.invoke("tell me about dogs")
print (response.content)

Dogs are great companions; they are especially known for their loyalty and friendliness.
